In [1]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, field_validator
from langchain_ollama import ChatOllama

In [2]:
from dotenv import load_dotenv
load_dotenv()
model = ChatOllama(model="llama3.2")

In [3]:
#Define the desired ds
class WeatherReport(BaseModel):
    city: str = Field(description="Name of the city")
    temperature: str = Field(description="Temperature of the city")
    #custom validation to ensure city name is not empty
    @field_validator("city")
    def city_name(cls, field):
        if not field.strip():
            raise ValueError("City name cannot be empty!")
        return field

In [6]:
#Setup a parse + inject intructions into the prompt template
parser = PydanticOutputParser(pydantic_object=WeatherReport)
prompt = PromptTemplate(
    template=(
        "Give a weather report for the requested city. "
        "Return only a JSON object with actual values, matching this example: "
        '{{"city": "London", "temperature": "15°C"}}\n'
        "{format_instructions}\n"
        "Request: {query}\n"
    ),
    input_variables=["query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

output = (prompt | model).invoke({"query": "What's the weather in London?"})
print(output.content)  # inspect the model's raw response
print(parser.invoke(output.content))

{"city": "London", "temperature": "12°C", "humidity": "60%", "wind": "10mph"}
city='London' temperature='12°C'
